**Question 1**: How long would it take to train a 70B parameter model on 15T tokens on 1024 H100s?

In [2]:
total_flops = 6 * 70e9 * 15e12  # @inspect total_flops
h100_flop_per_sec = 1979e12 / 2
mfu = 0.5
flops_per_day = h100_flop_per_sec * mfu * 1024 * 60 * 60 * 24  # @inspect flops_per_day
days = total_flops / flops_per_day  # @inspect day
print("flops per day: ",flops_per_day)
print("days: ",days)

flops per day:  4.37723136e+22
days:  143.92659381842682


**Question 2**: What's the largest model that can you can train on 8 H100s using AdamW?

In [5]:
h100_bytes = 80e9  # @inspect h100_bytes
bytes_per_parameter = 2 + 2 + (4 + 4)  # parameters (2), gradients (2), optimizer state (4 + 4) @inspect bytes_per_parameter
num_parameters = (h100_bytes * 8) / bytes_per_parameter  # @inspect num_parameters
print("bytes_per_parameter: ", bytes_per_parameter)
print("num_parameters: ", num_parameters)

bytes_per_parameter:  12
num_parameters:  53333333333.333336


#### Caveat: activations are not accounted for (depends on batch size and sequence length), so this is an upper bound.
This is a rough back-of-the-envelope calculation.
But it gives you the flavor of napkin math one can quickly do to get a sense of resources.


In [1]:
import torch
from torch import nn

In [2]:
def tensors_basics():
    print("Tensors are the basic building block for storing everything:")
    print("- data")
    print("- parameters")
    print("- gradients")
    print("- optimizer state")
    print("- activations")

    # print("Example: parameters of the DeepSeek v3.2 model "), link(deepseek_v3_2_2025)
    # link(title="DeepSeek v3.2 model on Hugging Face", url="https://huggingface.co/deepseek-ai/DeepSeek-V3.2?show_file_info=model.safetensors.index.json")

    print("Each tensor has a rank, which is the number of dimensions.")
    x = torch.zeros(4)        # rank 1 tensor (vector) @inspect x
    x = torch.zeros(4, 8)     # rank 2 tensor (matrix) @inspect x
    x = torch.zeros(4, 8, 2)  # rank 3 tensor @inspect x

    print("In Transformers, will see tensors of rank 4:")
    B = 32   # Batch size
    S = 16   # Sequence length
    H = 16   # Number of heads
    D = 64   # Hidden dimension per head
    x = torch.zeros(B, S, H, D)
tensors_basics()

Tensors are the basic building block for storing everything:
- data
- parameters
- gradients
- optimizer state
- activations
Each tensor has a rank, which is the number of dimensions.
In Transformers, will see tensors of rank 4:


In [3]:
torch.zeros(4,8)

tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [5]:
def get_memory_usage(x: torch.Tensor):
    return x.numel() * x.element_size()
    
def tensorMemory():
    x = torch.zeros(4, 8)  # @inspect x
    assert x.dtype == torch.float32  # Default type
    assert x.numel() == 4 * 8
    assert x.element_size() == 4  # Float is 4 bytes
    assert get_memory_usage(x) == 4 * 8 * 4  # 128 bytes

    print("One matrix in the feedforward layer of GPT-3:")
    assert get_memory_usage(torch.empty(12288 * 4, 12288)) == 2304 * 1024 * 1024  # 2.3 GB @stepover
tensorMemory()

One matrix in the feedforward layer of GPT-3:
